# OpenBench Phase 7: Final Held-Out Evaluation

This notebook reviews the final evaluation of the two saved IMDb sentiment models on the official labeled test split. It loads the generated evaluation outputs and verifies a small raw-text prediction sample against the saved model artifacts.

## 1. Phase 7 objectives

The goal is to report final test performance for the frozen TF-IDF + Logistic Regression and TextCNN artifacts. The official test data is used only for this evaluation. There is no training, tuning, checkpoint selection, or preprocessing fitting in this notebook.

## 2. Evaluation protocol and test-set isolation

The evaluation runner used the original test review directories and their deterministic loader order. Both models received the same 25,000 text records and labels. Logistic Regression reused its saved fitted vectorizer; TextCNN reused its saved checkpoint and training vocabulary. The full predictions are read from the Phase 7 CSV rather than regenerated here.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "pyproject.toml").is_file():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Could not locate the OpenBench project root.")
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

DATASET_DIRECTORY = PROJECT_ROOT / "datasets" / "aclImdb"
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "final_evaluation"
metadata_files = list(OUTPUT_ROOT.glob("imdb-final-evaluation-*/metadata.json"))
if not metadata_files:
    raise FileNotFoundError(
        "No Phase 7 results found. Run the final evaluation module first."
    )
METADATA_PATH = max(metadata_files, key=lambda path: path.stat().st_mtime)
FINAL_DIRECTORY = METADATA_PATH.parent
metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8"))
results = json.loads(
    (FINAL_DIRECTORY / "final_results.json").read_text(encoding="utf-8")
)
predictions = pd.read_csv(FINAL_DIRECTORY / "test_predictions.csv")
print(f"Results: {FINAL_DIRECTORY.relative_to(PROJECT_ROOT)}")
print(f"Python {sys.version.split()[0]} | PyTorch {torch.__version__}")

## 3. Load official test data

The existing loader reads only the labeled `test` split and preserves each source-relative record ID. The dataset contains auxiliary metadata files at the `test` directory root; they are not review records and the loader does not use them.

In [ ]:
from deeptext.data.dataset import load_imdb_split

test_records = load_imdb_split(DATASET_DIRECTORY, "test")
print(f"Loaded {len(test_records):,} labeled test reviews.")
print(test_records[["review_id", "label"]].head())

## 4. Validate dataset integrity

The loader rejects missing class directories, unexpected entries inside review directories, unreadable files, empty review text, and invalid labels. This check confirms the expected total, balanced class counts, unique test-relative IDs, and original row order.

In [ ]:
expected_distribution = {"negative": 12_500, "positive": 12_500}
actual_distribution = {
    "negative": int((test_records["label"] == 0).sum()),
    "positive": int((test_records["label"] == 1).sum()),
}
assert len(test_records) == 25_000
assert actual_distribution == expected_distribution
assert test_records["split"].eq("test").all()
assert test_records["review_id"].is_unique
assert (
    test_records["text"]
    .map(lambda text: isinstance(text, str) and bool(text.strip()))
    .all()
)
assert all(
    review_id.startswith("test/neg/")
    if label == 0
    else review_id.startswith("test/pos/")
    for review_id, label in zip(
        test_records["review_id"], test_records["label"], strict=True
    )
)
print(f"Class distribution: {actual_distribution}")
print(
    "Review ID order SHA256:",
    metadata["dataset_validation"]["review_id_order_sha256"],
)
auxiliary_files = metadata["dataset_validation"]["root_auxiliary_files_not_loaded"]
print(f"Root auxiliary files not loaded: {auxiliary_files}")

## 5. Load frozen Logistic Regression artifact

Load the exact Phase 3 run directory recorded in the evaluation metadata. The pipeline contains its already-fitted TF-IDF vectorizer and Logistic Regression classifier.

In [ ]:
from deeptext.models.baseline import load_baseline_artifacts

BASELINE_RUN = Path(metadata["artifact_paths"]["logistic_regression"])
baseline_model, baseline_metadata = load_baseline_artifacts(BASELINE_RUN)
assert (
    baseline_metadata["selected_settings"]
    == metadata["model_configurations"]["logistic_regression"]
)
print(baseline_metadata["model_name"], baseline_metadata["selected_settings"])

## 6. Load frozen TextCNN checkpoint

Load the saved best checkpoint with its training vocabulary. The notebook uses CUDA if available and otherwise uses CPU; no weights or vocabulary are fitted or changed.

In [ ]:
from deeptext.inference.predictor import load_textcnn_predictor

TEXTCNN_RUN = Path(metadata["artifact_paths"]["textcnn"])
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
textcnn_predictor = load_textcnn_predictor(TEXTCNN_RUN, device=DEVICE)
assert (
    textcnn_predictor.model.config.to_dict()
    == metadata["model_configurations"]["textcnn"]
)
assert textcnn_predictor.metadata["best_epoch"] == 2
best_epoch = textcnn_predictor.metadata["best_epoch"]
print(f"Loaded TextCNN on {DEVICE}; best epoch {best_epoch}.")

## 7. Generate Logistic Regression predictions

The complete aligned test predictions were produced by the Phase 7 runner. This cell performs a small first-row spot check through the existing saved pipeline, including class-1 probabilities.

In [ ]:
from deeptext.models.baseline import predict_baseline, predict_baseline_probabilities

spot_count = 32
baseline_spot_predictions = predict_baseline(
    baseline_model, test_records["text"].tolist()[:spot_count]
)
baseline_spot_probabilities = predict_baseline_probabilities(
    baseline_model, test_records["text"].tolist()[:spot_count]
)[:, 1]
assert (
    baseline_spot_predictions
    == predictions["logistic_regression_prediction"].tolist()[:spot_count]
)
assert np.allclose(
    baseline_spot_probabilities,
    predictions["logistic_regression_positive_probability"].to_numpy()[:spot_count],
    atol=1e-12,
)
print("Logistic Regression spot-check passed.")

## 8. Generate TextCNN predictions

Use the predictor's existing raw-text batch interface for a small spot check. Full test-set predictions remain those saved by the runner.

In [ ]:
textcnn_spot = textcnn_predictor.predict_batch(
    test_records["text"].tolist()[:spot_count], batch_size=16
)
assert [row["predicted_class"] for row in textcnn_spot] == predictions[
    "textcnn_prediction"
].tolist()[:spot_count]
assert np.allclose(
    [row["positive_probability"] for row in textcnn_spot],
    predictions["textcnn_positive_probability"].to_numpy()[:spot_count],
    atol=1e-6,
)
print("TextCNN predictions and positive-class probabilities match the saved rows.")

## 9. Verify prediction alignment

Compare the saved prediction IDs and true labels directly with the dataset loader output. Each model has exactly one prediction and class-1 probability per official test record.

In [ ]:
labels = test_records["label"].to_numpy(dtype=np.int64)
assert len(predictions) == len(test_records) == 25_000
assert predictions["review_id"].tolist() == test_records["review_id"].tolist()
assert np.array_equal(predictions["true_label"].to_numpy(), labels)
assert predictions["logistic_regression_prediction"].isin([0, 1]).all()
assert predictions["textcnn_prediction"].isin([0, 1]).all()
assert np.isfinite(predictions["logistic_regression_positive_probability"]).all()
assert np.isfinite(predictions["textcnn_positive_probability"]).all()
assert metadata["artifacts_unchanged"] is True
print("Both prediction arrays align with the same 25,000 ordered test records.")

## 10. Calculate classification metrics

Recalculate the shared metrics from the saved full-test predictions. Labels use negative `0` and positive `1`; confusion rows and columns are ordered `[0, 1]`; zero-division behavior is `0`.

In [ ]:
from deeptext.evaluation.comparison import evaluate_comparison_model

recalculated_metrics = {
    "logistic_regression": evaluate_comparison_model(
        labels,
        predictions["logistic_regression_prediction"],
        predictions["logistic_regression_positive_probability"],
    ),
    "textcnn": evaluate_comparison_model(
        labels,
        predictions["textcnn_prediction"],
        predictions["textcnn_positive_probability"],
    ),
}
for model_name, measured in recalculated_metrics.items():
    for metric_name in ("accuracy", "precision", "recall", "f1"):
        assert np.isclose(
            measured[metric_name],
            results["metrics"][model_name][metric_name],
            rtol=0,
            atol=1e-12,
        )
    assert (
        measured["confusion_matrix"]
        == results["metrics"][model_name]["confusion_matrix"]
    )
print("Recalculated metrics and confusion matrices match final_results.json.")

## 11. Compare the two models

The table reports all 25,000 official test examples. The scores describe these frozen runs and this dataset split.

In [ ]:
metric_rows = []
for model_key, model_name in [
    ("logistic_regression", "TF-IDF + Logistic Regression"),
    ("textcnn", "TextCNN"),
]:
    values = results["metrics"][model_key]
    metric_rows.append(
        {
            "Model": model_name,
            "Accuracy (%)": 100 * values["accuracy"],
            "Precision (%)": 100 * values["precision"],
            "Recall (%)": 100 * values["recall"],
            "F1 (%)": 100 * values["f1"],
            "False positives": values["false_positive"],
            "False negatives": values["false_negative"],
        }
    )
display(pd.DataFrame(metric_rows).set_index("Model").round(2))

## 12. Validation versus test analysis

The validation values come from the Phase 3 and Phase 5 saved metadata; the test values are newly measured here. The displayed difference is **validation − test**, in percentage points. A negative value means the test metric is higher. This observed difference is not a complete measure of generalization.

In [ ]:
validation_test = pd.DataFrame(results["validation_vs_test"])
validation_test["Validation (%)"] = 100 * validation_test["validation"]
validation_test["Test (%)"] = 100 * validation_test["test"]
validation_test["Validation − test (pp)"] = validation_test[
    "difference_percentage_points"
]
display(
    validation_test[
        ["metric", "model", "Validation (%)", "Test (%)", "Validation − test (pp)"]
    ].round(3)
)

## 13. Confusion matrices

Rows show actual sentiment and columns show predicted sentiment, in negative-then-positive order.

In [ ]:
display(
    Image(filename=str(FINAL_DIRECTORY / "confusion_matrix_logistic_regression.png"))
)
display(Image(filename=str(FINAL_DIRECTORY / "confusion_matrix_textcnn.png")))

## 14. Error analysis

Counts cover the entire official test split. The CSV contains only selected examples with their original test IDs and review snippets capped at 400 characters. The model probabilities are each model's own outputs and are not assumed to be calibrated against one another.

In [ ]:
error_counts = results["error_analysis"]["counts"]
display(
    pd.DataFrame(
        [
            {"Error category": name.replace("_", " ").capitalize(), "Count": count}
            for name, count in error_counts.items()
        ]
    )
)
error_examples = pd.read_csv(FINAL_DIRECTORY / "error_analysis.csv")
display(error_examples.head(20))
print(results["error_analysis"]["example_selection"])

## 15. Visualizations

All figures use Matplotlib and are loaded from the generated output directory.

In [ ]:
for plot_name in [
    "classification_metrics.png",
    "validation_vs_test_comparison.png",
    "generalization_gap.png",
    "false_positive_negative.png",
    "prediction_agreement.png",
]:
    display(Image(filename=str(FINAL_DIRECTORY / plot_name)))

## 16. Final observations

Logistic Regression has higher test accuracy, precision, and F1 in this run. TextCNN has slightly higher recall and fewer false negatives, with more false positives. The test scores are close to the saved validation scores for both models, but they are one final evaluation and do not establish statistical significance or universal superiority.

## 17. Limitations and conclusion

The original validation artifacts did not retain row IDs or fingerprints. Phase 6 reconstructed the common validation split and exactly reproduced both recorded metric sets, but historical row membership cannot be independently proven from those artifacts. That reconstructed validation split contained 26 unique exact-text duplicates shared with training across 28 validation rows. This may influence validation scores; the evidence does not show that it caused any particular validation-to-test difference.

No model training, hyperparameter tuning, checkpoint selection, or preprocessing fitting occurred in Phase 7. The official test split was evaluated once for final reporting. Results describe these artifacts, this dataset, and this environment. The test set must not be used to tune future models.